In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import torch
import math
import torch.nn as nn
import torch.nn.functional as F
from timm.models.layers import to_2tuple, trunc_normal_
from collections import OrderedDict
from torch.utils.data import DataLoader
from torchvision import transforms
import shutil
from datasets import load_dataset
from PIL import Image

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
def rounding_shift_right(tensor, shift_amt):
    sign = tensor.sign()
    tensor = tensor.abs()
    mask = (2 ** shift_amt.abs().to(torch.int32)) - 1
    lsb = tensor & mask
    tensor = tensor >> shift_amt.abs()

    threshold = (mask >> 1) + 1
    is_greater_than_half = lsb > threshold
    is_half_and_odd = (lsb == threshold) & ((tensor & 1) == 1)
    should_round_up = is_greater_than_half | is_half_and_odd
    tensor = torch.where(
            should_round_up,
            tensor + 1,
            tensor
        )
    tensor = tensor * sign

    return tensor

def shift_left(tensor, shift_amt):
    tensor = tensor << shift_amt
    return tensor


def quantize(tensor: torch.Tensor, scale: torch.Tensor, is_input=True) -> torch.Tensor:

    Q_MIN = -127
    Q_MAX = 127
    scale_view = scale.squeeze()

    if is_input:
        quantized_tensor = tensor / scale_view

        quantized_tensor = torch.round(quantized_tensor)

        return quantized_tensor.to(torch.int8)
    else:

        tensor = torch.round(tensor).to(torch.int32)
        quantized_tensor = torch.where(
            scale_view > 0,
            rounding_shift_right(tensor, scale_view.abs()),
            shift_left(tensor, scale_view.abs())
        )
        quantized_tensor = torch.round(quantized_tensor)
        quantized_tensor = torch.clamp(quantized_tensor, Q_MIN, Q_MAX)

        return quantized_tensor.to(torch.int8)


def dequantize(quantized_tensor: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:

    Q_MIN = -127
    Q_MAX = 127
    scale_view = scale.squeeze()

    dequantized_tensor = quantized_tensor * scale_view
    return dequantized_tensor

def log2_quantize(tensor):

    log2_tensor = torch.round(torch.log2(tensor)).abs()

    log2_tensor = torch.clamp(log2_tensor, 0, 127).to(torch.int8)

    power = 7 - log2_tensor

    output = 1 << power.to(torch.int32)

    output = torch.round(output)
    output = torch.clamp(output, -127, 127)

    return output.to(torch.int8)

In [ ]:
def softmax(x, dim=-1):

    x = x.float()

    max_x = torch.max(x, dim=dim, keepdim=True)[0]

    x = x - max_x
    x = torch.clamp(x, -127, 127).to(torch.int8)

    two_x = torch.where(
            x>=-4,
            torch.pow(2.0, x),
            0
        )

    sum_two_x = torch.sum(two_x, dim=dim, keepdim=True)

    output = two_x / sum_two_x

    return output

class FloatAttnMaskSoftmax(nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        # self.num_heads = num_heads

    def forward(self, attn, mask, B_, N):
        if mask is not None:
            nW = mask.shape[1]

            attn = attn.view(B_ // nW, nW, 1, N, N)

            attn = attn + mask

            attn = attn.view(-1, 1, N, N)

        attn = softmax(attn)
        return attn

In [ ]:
def conv2d(x_int8, weight_int8, bias_int32, stride=1):

    batch_size = x_int8.shape[0]

    outputs = []
    for b in range(batch_size):
        res = torch._int_mm(weight_int8, x_int8[b])
        if bias_int32 is not None:
            res += bias_int32
        outputs.append(res)

    output_tensor = torch.stack(outputs).view(batch_size, 96, 56, 56)

    return output_tensor

def linear(x_int8, weight_int8, bias_int32=None):
    original_shape = x_int8.shape
    x_i8 = x_int8.reshape(-1, original_shape[-1]).to(torch.int8)
    w_i8 = weight_int8.to(torch.int8).t()

    output_matmul = torch._int_mm(x_i8, w_i8)

    if bias_int32 is not None:
        output_matmul = output_matmul + bias_int32

    new_shape = list(original_shape[:-1]) + [weight_int8.shape[0]]

    final_output = output_matmul.reshape(new_shape)

    return final_output

In [ ]:
def matmul_int8_4d(x_int8, y_int8):

    B, H, N, K = x_int8.shape
    D = y_int8.shape[-1]

    x_flat = x_int8.reshape(-1, N, K).to(torch.int8)
    y_flat = y_int8.reshape(-1, K, D).to(torch.int8)

    D_padded = (D + 7) // 8 * 8
    pad_D = D_padded - D

    K_padded = (K + 7) // 8 * 8
    pad_K = K_padded - K

    if pad_D > 0 or pad_K > 0:
        x_flat = torch.nn.functional.pad(x_flat, (0, pad_K, 0, 0))
        y_flat = torch.nn.functional.pad(y_flat, (0, pad_D, 0, pad_K))

    results = []
    for i in range(B * H):
        res = torch._int_mm(x_flat[i], y_flat[i])

        if pad_D > 0:
            res = res[:, :D]
        results.append(res)

    output = torch.stack(results).view(B, H, N, D)
    return output

In [ ]:
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None,
                 out_features=None, act_layer=nn.ReLU):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features

        self.register_buffer("fuse_linear_bn1_weights", None)
        self.register_buffer("fuse_linear_bn1_bias", None)
        self.register_buffer("fuse_linear_bn1_output_scale", None)

        self.register_buffer("fuse_linear_bn2_weights", None)
        self.register_buffer("fuse_linear_bn2_bias", None)
        self.register_buffer("fuse_linear_bn2_output_scale", None)

    def forward(self, x):

        x = linear(x, self.fuse_linear_bn1_weights, self.fuse_linear_bn1_bias)

        x = quantize(x, self.fuse_linear_bn1_output_scale, is_input=False)

        x = nn.ReLU()(x)

        x = linear(x, self.fuse_linear_bn2_weights, self.fuse_linear_bn2_bias)

        x = quantize(x, self.fuse_linear_bn2_output_scale, is_input=False)

        return x

def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)

    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)

    return windows


def window_reverse(windows, window_size, H, W):
    B = windows.shape[0] // (H * W // window_size // window_size)

    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)

    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)

    return x


class WindowAttention(nn.Module):

    def __init__(self, dim, window_size, num_heads, qkv_bias=True):

        super().__init__()
        self.dim = dim
        self.window_size = window_size  # Wh, Ww
        self.num_heads = num_heads
        head_dim = dim // num_heads

        self.float_mask_softmax = FloatAttnMaskSoftmax(num_heads)

        self.register_buffer("q_weight", None)
        self.register_buffer("q_bias", None)
        self.register_buffer("q_output_scale", None)

        self.register_buffer("k_weight", None)
        self.register_buffer("k_bias", None)
        self.register_buffer("k_output_scale", None)

        self.register_buffer("relative_position_bias", None)
        self.register_buffer("qk_output_scale", None)


        self.register_buffer("v_weight", None)
        self.register_buffer("v_bias", None)
        self.register_buffer("v_output_scale", None)

        self.register_buffer("attn_v_output_scale", None)

        self.register_buffer("proj_weights", None)
        self.register_buffer("proj_bias", None)
        self.register_buffer("proj_output_scale", None)


    def forward(self, x, mask):

        B_, N, C = x.shape

        q = linear(x, self.q_weight, self.q_bias)
        k = linear(x, self.k_weight, self.k_bias)
        v = linear(x, self.v_weight, self.v_bias)

        q = q.reshape(B_, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        k = k.reshape(B_, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        v = v.reshape(B_, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)

        # Split into per-head variables — each: (B_, N, C // num_heads)
        q_heads = [q[:, i, :, :] for i in range(self.num_heads)]
        k_heads = [k[:, i, :, :] for i in range(self.num_heads)]
        v_heads = [v[:, i, :, :] for i in range(self.num_heads)]

        head_dim = C // self.num_heads

        out_heads = []
        for i in range(self.num_heads):
            q_i = quantize(q_heads[i], self.q_output_scale[i * head_dim : (i+1) * head_dim], is_input=False).unsqueeze(1)  # (B_, 1, N, head_dim)
            k_i = quantize(k_heads[i], self.k_output_scale[i * head_dim : (i+1) * head_dim], is_input=False).unsqueeze(1)  # (B_, 1, N, head_dim)

            # Attention scores
            attn_i = matmul_int8_4d(q_i, k_i.transpose(-2, -1))  # (B_, 1, N, N)

            # Add relative position bias for this head
            attn_i = attn_i + self.relative_position_bias[:, i:i+1, :, :]

            attn_i = quantize(attn_i, self.qk_output_scale, is_input=False)

            attn_i = self.float_mask_softmax(attn_i, mask, B_, N)

            attn_i = log2_quantize(attn_i)

            v_i = quantize(v_heads[i], self.v_output_scale[i * head_dim : (i+1) * head_dim], is_input=False).unsqueeze(1)  # (B_, 1, N, head_dim)

            # Weighted sum over values
            out_i = matmul_int8_4d(attn_i, v_i)  # (B_, 1, N, C // num_heads)

            out_i = quantize(out_i, self.attn_v_output_scale, is_input=False)

            out_heads.append(out_i)

        # Recombine heads
        x = torch.cat(out_heads, dim=1)          # (B_, num_heads, N, C // num_heads)

        x = x.transpose(1, 2).reshape(B_, N, C)  # (B_, N, C)

        x = linear(x, self.proj_weights, self.proj_bias)

        x = quantize(x, self.proj_output_scale, is_input=False)

        return x

class SwinTransformerBlock(nn.Module):

    def __init__(self, dim, input_resolution, num_heads, window_size=7, shift_size=0,
                 mlp_ratio=4., qkv_bias=True, act_layer=nn.ReLU):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        if min(self.input_resolution) <= self.window_size:
            # if window size is larger than input resolution, we don't partition windows
            self.shift_size = 0
            self.window_size = min(self.input_resolution)
        assert 0 <= self.shift_size < self.window_size, "shift_size must in 0-window_size"

        self.attn = WindowAttention(
            dim, window_size=to_2tuple(self.window_size), num_heads=num_heads,
            qkv_bias=qkv_bias)

        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer)

        self.register_buffer("attn_mask", None)

    def forward(self, x):
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"

        shortcut = x

        x = x.view(B, H, W, C)

        # cyclic shift
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))

            # partition windows
            x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C
        else:
            shifted_x = x
            # partition windows
            x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C

        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)  # nW*B, window_size*window_size, C


        attn_windows = self.attn(x_windows, self.attn_mask)  # nW*B, window_size*window_size, C

        # merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)

        # reverse cyclic shift
        if self.shift_size > 0:
            shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))

        else:
            shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
            x = shifted_x

        x = x.view(B, H * W, C)

        x = shortcut + x

        x_mlp = self.mlp(x)

        x = x + x_mlp

        return x


class PatchMerging(nn.Module):

    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim

        self.register_buffer("input_norm_weights", None)
        self.register_buffer("input_norm_bias", None)

        self.register_buffer("input_norm_output_scale", None)

    def forward(self, x):

        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"
        assert H % 2 == 0 and W % 2 == 0, f"x size ({H}*{W}) are not even."


        x = x.view(B, H, W, C)

        x0 = x[:, 0::2, 0::2, :]  # B H/2 W/2 C

        x1 = x[:, 1::2, 0::2, :]  # B H/2 W/2 C

        x2 = x[:, 0::2, 1::2, :]  # B H/2 W/2 C

        x3 = x[:, 1::2, 1::2, :]  # B H/2 W/2 C

        x = torch.cat([x0, x1, x2, x3], -1)  # B H/2 W/2 4*C

        x = x.view(B, -1, 4 * C)  # B H/2*W/2 4*C

        x = linear(x, self.input_norm_weights, self.input_norm_bias)

        x = quantize(x, self.input_norm_output_scale, is_input=False)

        return x


class BasicLayer(nn.Module):

    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, downsample=None):

        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.depth = depth

        # build blocks
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(dim=dim, input_resolution=input_resolution,
                                 num_heads=num_heads, window_size=window_size,
                                 shift_size=0 if (i % 2 == 0) else window_size // 2,
                                 mlp_ratio=mlp_ratio,
                                 qkv_bias=qkv_bias)
            for i in range(depth)])

        # patch merging layer
        if downsample is not None:
            self.downsample = downsample(input_resolution, dim=dim)
        else:
            self.downsample = None

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x


class PatchEmbed(nn.Module):

    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        patches_resolution = [img_size[0] // patch_size[0], img_size[1] // patch_size[1]]
        self.img_size = img_size
        self.patch_size = patch_size
        self.patches_resolution = patches_resolution

        self.in_chans = in_chans
        self.embed_dim = embed_dim

        self.register_buffer("fused_proj_weights", None)
        self.register_buffer("fused_proj_bias", None)

        self.register_buffer("fused_proj_output_scale", None)

    def forward(self, x):
        B, C, H, W = x.shape
        # FIXME look at relaxing size constraints
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."

        # 96 * 48
        w_int8 = self.fused_proj_weights.view(self.embed_dim, -1)

        # 96 * 1
        b_int32 = self.fused_proj_bias.view(-1, 1)

        # 3 * 224 * 224 -> 48 * 3136
        x = torch.nn.functional.unfold(x.float(), kernel_size=(self.patch_size[0], self.patch_size[1]), stride=self.patch_size[0]).to(torch.int8)#software

        # 96 * 3136
        x = conv2d(x, w_int8, b_int32, stride=self.patch_size[0])

        # 96 * 56 * 56 -> 96 * 3136 -> 3136 * 96
        x = x.flatten(2).transpose(1, 2)

        x = quantize(x, self.fused_proj_output_scale, is_input=False)

        return x

def avg_pool(x):
    length = x.shape[2] # 49
    sum_x = torch.sum(x, dim=2, keepdim=True)

    output = sum_x / length

    return output


class SwinTransformer(nn.Module):

    def __init__(self, img_size=224, patch_size=4, in_chans=3, num_classes=1000,
                 embed_dim=96, depths=[2, 2, 6, 2], num_heads=[3, 6, 12, 24],
                 window_size=7, mlp_ratio=4., qkv_bias=True, **kwargs):
        super().__init__()

        self.num_classes = num_classes
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))
        self.mlp_ratio = mlp_ratio

        # split image into non-overlapping patches
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim)
        patches_resolution = self.patch_embed.patches_resolution
        self.patches_resolution = patches_resolution

        # build layers
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            layer = BasicLayer(dim=int(embed_dim * 2 ** i_layer),
                               input_resolution=(patches_resolution[0] // (2 ** i_layer),
                                                 patches_resolution[1] // (2 ** i_layer)),
                               depth=depths[i_layer],
                               num_heads=num_heads[i_layer],
                               window_size=window_size,
                               mlp_ratio=self.mlp_ratio,
                               qkv_bias=qkv_bias,
                               downsample=PatchMerging if (i_layer < self.num_layers - 1) else None)
            self.layers.append(layer)

        self.register_buffer("input_scale", None)

        self.register_buffer("input_norm_weights", None)
        self.register_buffer("input_norm_bias", None)
        self.register_buffer("input_norm_output_scale", None)

        self.head = nn.Sequential(OrderedDict([("fc", nn.Linear(self.num_features, num_classes))])) if num_classes > 0 else nn.Identity()
        self.register_buffer("output_scale", None)

    def forward_features(self, x):
        # 3 * 224 * 224
        x = quantize(x, self.input_scale)#software

        x = self.patch_embed(x)

        for layer in self.layers:
            x = layer(x)

        x = linear(x, self.input_norm_weights, self.input_norm_bias)

        x = quantize(x, self.input_norm_output_scale, is_input=False)

        x = x.to(torch.float32)

        x = x.transpose(1, 2)#software

        x = avg_pool(x)#software  # B C 1

        x = torch.flatten(x, 1)#software

        return x

    def forward(self, x):
        x = self.forward_features(x)

        x = self.head(x)#software

        x = dequantize(x, self.output_scale)#software

        return x

In [ ]:
# 2. Create your custom implementation
custom_model = SwinTransformer(
    img_size=224,
    patch_size=4,
    in_chans=3,
    num_classes=1000,
    embed_dim=96,
    depths=[2, 2, 6, 2],
    num_heads=[3, 6, 12, 24],
)

In [ ]:
orig_state = torch.load("/content/drive/MyDrive/final_weights15.pth")

In [ ]:
for name, param in orig_state.items():
    # Check if the name corresponds to an observer's scale
    if "scale" in name or "relative_position_bias" in name or "shift" in name or "attn_mask" in name or "_weight" in name or "_bias" in name and "table" not in name:
        # Construct the path to the observer module
        parts = name.split('.')
        current_module = custom_model
        module_found = True
        for i, part in enumerate(parts[:-1]):
            if hasattr(current_module, part):
                current_module = getattr(current_module, part)
            else:
                # If a part of the path doesn't exist, this key is not applicable
                module_found = False
                break

        if module_found:
            attr_name = parts[-1] # This should be 'scale'
            if hasattr(current_module, attr_name):
                # Directly assign the parameter from the loaded state to the model's buffer
                # This will resize the buffer if necessary, fixing the shape mismatch.
                with torch.no_grad(): # Ensure this operation doesn't track gradients
                    # Ensure device consistency by moving param to buffer's device if different
                    setattr(current_module, attr_name, param)
            else:
                print(f"Warning: Attribute '{attr_name}' not found in module '{'.'.join(parts[:-1])}' for key '{name}'. Skipping.")
        else:
            print(f"Warning: Module path for key '{name}' not fully resolvable. Skipping.")


# Load into your model
missing, unexpected = custom_model.load_state_dict(orig_state, strict=False)


# Optionally inspect for any mismatches
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

Missing keys: []
Unexpected keys: []


In [ ]:
!hf auth login

A new version of huggingface_hub (1.5.0) is available! You are using version 1.4.1.
To update, run: pip install -U huggingface_hub


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: 
Token is valid (permission: fineGrained).
The token `swin` has been saved to /root/.cache/huggingface/sto

In [ ]:
# ----------------------------------------------------------------------------
# 1) Load ImageNet-1k training set (streaming mode)
# ----------------------------------------------------------------------------
hf_dataset_train = load_dataset("ILSVRC/imagenet-1k", split="train", streaming=True)
hf_dataset_val = load_dataset("ILSVRC/imagenet-1k", split="validation", streaming=True)


train_stream = hf_dataset_train.take(100_000)
val_stream   = hf_dataset_train.skip(100_000).take(15_000)
test_stream = hf_dataset_val.take(10_000)


# ----------------------------------------------------------------------------
# 2) Preprocessing pipeline
# ----------------------------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def collate_fn(batch):
    """Convert HF samples into tensors"""
    images, labels = [], []
    for sample in batch:
        img = sample["image"]
        if not isinstance(img, Image.Image):  # if numpy array
            img = Image.fromarray(img)
        img = img.convert("RGB")  # <-- ensure 3 channels
        images.append(transform(img))
        labels.append(sample["label"])
    return torch.stack(images), torch.tensor(labels)


# ----------------------------------------------------------------------------
# 3) Wrap datasets with DataLoader
# ----------------------------------------------------------------------------
train_loader = DataLoader(train_stream, batch_size=64, collate_fn=collate_fn)
val_loader   = DataLoader(val_stream,   batch_size=64, collate_fn=collate_fn)
test_loader  = DataLoader(test_stream, batch_size=64, collate_fn=collate_fn)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    top1, top5, total, total_loss = 0, 0, 0, 0.0

    with torch.no_grad():
        for i, (images, labels) in enumerate(dataloader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Top-1
            _, pred1 = outputs.topk(1, dim=1)
            top1 += (pred1.squeeze() == labels).sum().item()

            # Top-5
            _, pred5 = outputs.topk(5, dim=1)
            top5 += sum([labels[j].item() in pred5[j] for j in range(labels.size(0))])

            total += labels.size(0)

            if (i+1) % 1 == 0:
                print(f"[Val Step {i+1}] "
                      f"Loss: {total_loss/(i+1):.4f} | "
                      f"Top-1: {100.*top1/total:.2f}% | "
                      f"Top-5: {100.*top5/total:.2f}%")

    return 100.*top1/total, 100.*top5/total, total_loss/(i+1)

# ----------------------------------------------------------------------------
# 5) Train the model
# ----------------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
custom_model.to(device)

criterion = nn.CrossEntropyLoss()

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

In [ ]:
test_top1, test_top5, test_loss = evaluate(
    custom_model, train_loader, criterion, device
)
print(f"\nPTQ Model Results:")
print(f"Top-1: {test_top1:.2f}% | Top-5: {test_top5:.2f}%")

[Val Step 1] Loss: 0.4704 | Top-1: 92.19% | Top-5: 96.88%
[Val Step 2] Loss: 0.5810 | Top-1: 89.06% | Top-5: 96.88%
[Val Step 3] Loss: 0.6019 | Top-1: 88.02% | Top-5: 96.88%
[Val Step 4] Loss: 0.5620 | Top-1: 88.67% | Top-5: 97.66%


KeyboardInterrupt: 

In [ ]:
test_top1, test_top5, test_loss = evaluate(
    custom_model, test_loader, criterion, device
)
print(f"\nPTQ Model Results:")
print(f"Top-1: {test_top1:.2f}% | Top-5: {test_top5:.2f}%")

[Val Step 1] Loss: 1.0925 | Top-1: 78.12% | Top-5: 89.06%
[Val Step 2] Loss: 1.2648 | Top-1: 73.44% | Top-5: 86.72%
[Val Step 3] Loss: 1.2471 | Top-1: 73.44% | Top-5: 87.50%
[Val Step 4] Loss: 1.2649 | Top-1: 72.27% | Top-5: 87.89%
[Val Step 5] Loss: 1.2572 | Top-1: 72.19% | Top-5: 88.12%
[Val Step 6] Loss: 1.3002 | Top-1: 71.61% | Top-5: 87.50%
[Val Step 7] Loss: 1.3136 | Top-1: 72.10% | Top-5: 87.72%


KeyboardInterrupt: 

In [ ]:
torch.save(custom_model.state_dict(), "final_weights15.pth")

drive_path = "/content/drive/MyDrive/final_weights15.pth"
shutil.copy("final_weights15.pth", drive_path)

'/content/drive/MyDrive/final_weights15.pth'

In [ ]:
def process_scale_keys(input_path, output_path):
    # 1. تحميل ملف الـ checkpoint
    checkpoint = torch.load(input_path, map_location='cpu')

    # 2. التحقق مما إذا كان الملف قاموساً مباشراً أو يحتوي على 'state_dict'
    # غالباً ملفات الـ pth تكون عبارة عن state_dict
    state_dict = checkpoint.get('state_dict', checkpoint)

    processed_count = 0

    # 3. تعديل القيم المطلوبة
    for key in state_dict.keys():
        if 'scale' in key.lower() and key.lower() != 'input_scale' and key.lower() != 'output_scale':
            # التأكد من أن القيمة عبارة عن Tensor
            value = state_dict[key]

            # حساب log2 ثم التحويل إلى int8
            # ملاحظة: نستخدم clamp أو rounding إذا لزم الأمر قبل التحويل
            log_scaled = torch.log2(value)
            state_dict[key] = log_scaled.to(torch.int8)

            processed_count += 1
            print(f"Updated key: {key}")

    # 4. حفظ الملف الجديد
    torch.save(checkpoint, output_path)
    print(f"\nDone! Processed {processed_count} keys.")
    print(f"Saved to: {output_path}")

# استخدام الكود
process_scale_keys('final_weights12.pth', 'final_weights13.pth')

Updated key: input_norm_output_scale
Updated key: patch_embed.fused_proj_output_scale
Updated key: layers.0.blocks.0.attn.q_output_scale
Updated key: layers.0.blocks.0.attn.k_output_scale
Updated key: layers.0.blocks.0.attn.v_output_scale
Updated key: layers.0.blocks.0.attn.qk_output_scale
Updated key: layers.0.blocks.0.attn.attn_v_output_scale
Updated key: layers.0.blocks.0.attn.proj_output_scale
Updated key: layers.0.blocks.0.mlp.fuse_linear_bn1_output_scale
Updated key: layers.0.blocks.0.mlp.fuse_linear_bn2_output_scale
Updated key: layers.0.blocks.1.attn.q_output_scale
Updated key: layers.0.blocks.1.attn.k_output_scale
Updated key: layers.0.blocks.1.attn.v_output_scale
Updated key: layers.0.blocks.1.attn.qk_output_scale
Updated key: layers.0.blocks.1.attn.attn_v_output_scale
Updated key: layers.0.blocks.1.attn.proj_output_scale
Updated key: layers.0.blocks.1.mlp.fuse_linear_bn1_output_scale
Updated key: layers.0.blocks.1.mlp.fuse_linear_bn2_output_scale
Updated key: layers.0.downsa

In [ ]:
orig_state = torch.load("/content/drive/MyDrive/final_weights15.pth")
with open("final_weights15.txt", "w") as f:
    for name, param in orig_state.items():
        f.write(f"{name}: {param}\n\n")

drive_path = "/content/drive/MyDrive/final_weights15.txt"
shutil.copy("final_weights15.txt", drive_path)

'/content/drive/MyDrive/final_weights15.txt'